In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle
from sklearn.metrics import roc_auc_score


In [2]:
df_saudi = pd.read_excel("Employee attrition dataset for tree-based models.xlsx")

In [3]:
columns_to_delete = ["Maritalstatus", "Department", "JobTitle", "Allowances"]
# allowances and marital status dropped since legend not clear 
# department and jobtitle dropped since we predict attrition for each sector, want enough data so we aggregate over the departments/jobs

df_saudi = df_saudi.drop(columns=columns_to_delete)

#change name of target variable to "target_saudi"
df_saudi = df_saudi.rename(columns={"Attrition": "target_saudi"})

In [4]:
# Convert categorical string codes to integers
# 0.565187=1, 0.359278=2, 0.260830=3, 0.456738=4, 0.512493=5
sector_map = {0.5651866011647716: 1, 0.3592782301026135: 2, 0.2608303362988201: 3, 0.4567380973729042: 4, 0.5124929017603634: 5}

df_saudi['Sector'] = df_saudi['Sector'].map(sector_map)


In [5]:
df_saudi.shape
df_saudi.head()

,target_saudi,Gender,Age,Academic_degree,Years_Experience,Years_experience_lastorganization,Sector,MonthlySalary,MedicalInsurance,Bonus,...,Job_Engagement,Distance_to_work,Work_Live_Balance,Physical_Stress,Psychological_Exhaustion,Job_Stability,Health_Issues,Environment_Satisfaction,Job_Satisfaction,Job_Opportunities
0,1,0,1,2,0,0,2,1,0,0,...,1,1,1,2,0,0,1,1,0,1
1,0,0,0,2,0,0,2,2,1,1,...,1,1,1,0,0,1,0,1,1,0
2,0,0,0,1,0,0,3,0,0,0,...,1,0,1,2,1,0,0,1,1,1
3,0,0,0,1,0,0,3,0,0,0,...,1,1,1,2,2,0,0,0,0,0
4,0,0,0,1,0,0,3,0,0,0,...,1,1,1,1,1,0,0,1,0,0


In [6]:
# define protected attributes 
protected_attributes = ["Gender", "Age", "Health_Issues"]


In [7]:
random_state = 1234
saudi_train, saudi_test = train_test_split(df_saudi, test_size=0.2, random_state=random_state)

saudi_train.to_parquet("train_cleaned.parquet")
saudi_test.to_parquet("test_cleaned.parquet")

# Separate features and target
x_train = saudi_train.drop("target_saudi", axis=1)
y_train = saudi_train["target_saudi"]
x_test = saudi_test.drop("target_saudi", axis=1)
y_test = saudi_test["target_saudi"]


In [8]:
# random forest model WITHOUT protected attributes 

# Create training and test sets without protected attributes for the model
x_train_model = x_train.drop(columns=protected_attributes)
x_test_model = x_test.drop(columns=protected_attributes)

rf_model = RandomForestClassifier(random_state=random_state, class_weight='balanced')
rf_model.fit(x_train_model, y_train)

# evaluate the model on accuracy and AUC metrics
rf_pred = rf_model.predict(x_test_model)
rf_pred_proba = rf_model.predict_proba(x_test_model)[:, 1]  # Get probability for class 1
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")
rf_auc = roc_auc_score(y_test, rf_pred_proba)
print(f"AUC: {rf_auc * 100:.2f}%")

# Save the model
with open('RF.pkl', 'wb') as f:
    pickle.dump(rf_model, f)    

Accuracy: 82.55%
AUC: 90.48%


In [9]:
# Define feature types for saudi dataset
numerical_features = ['Experience', 'Years_in_Job', 'Training_Programs']  # Only these are numeric
categorical_features = [col for col in x_train.columns if col not in numerical_features]

# Define feature value mappings for readable display
FEATURE_MAPPINGS = {
    'Gender': {0: 'Female', 1: 'Male'},
    'Age': {0: '21-30', 1: '31-40', 3: '41+'},
    'Education': {0: 'secondary school', 1: 'bachelor', 2: 'master', 3: 'PhD'},
    'Experience': {0: '1-5 years', 1: '6-10 years', 2: '11+ years'},
    'Years_in_Job': {0: '1-5 years', 1: '6-10 years', 2: '11+ years'},
    'Sector': {1: 'other', 2: 'medical', 3: 'education', 4: 'financial', 5: 'food'},
    'Salary': {0: '1k-5k SAR', 1: '6k-10k SAR', 2: '11k-15k SAR', 3: '16k+ SAR'},
    'Medical_Insurance': {0: 'no', 1: 'yes'},
    'Annual_Bonus': {0: 'no', 1: 'yes'},
    'Overtime': {0: 'no', 1: 'yes'},
    'Overtime_Compensation': {0: 'no overtime', 1: 'no', 2: 'yes'},
    'Income_Satisfaction': {0: 'no', 1: 'yes'},
    'Promotion_Satisfaction': {0: 'no', 1: 'yes'},
    'Training_Programs': {0: 'none', 1: '1-3', 2: '4-6', 3: '7+'},
    'Training_Benefit': {0: 'no', 1: 'yes'},
    'Business_Travel': {0: 'never', 1: 'rarely', 2: 'frequently'},
    'Organizational_Support': {0: 'low', 1: 'medium', 2: 'high'},
    'Moral_Appreciation': {0: 'no', 1: 'yes'},
    'Organizational_Commitment': {0: 'low', 1: 'medium', 2: 'high'},
    'Work_Involvement': {0: 'easy', 1: 'medium', 2: 'difficult'},
    'Distance_to_Workplace': {0: 'close', 1: 'medium', 2: 'far'},
    'Work_Life_Balance': {0: 'easy', 1: 'medium', 2: 'difficult'},
    'Physical_Stress': {0: 'no', 1: 'sometimes', 2: 'yes'},
    'Emotional_Exhaustion': {0: 'no', 1: 'sometimes', 2: 'yes'},
    'Job_Security': {0: 'no', 1: 'yes'},
    'Health_Issues': {0: 'no', 1: 'yes'},
    'Work_Environment_Satisfaction': {0: 'low', 1: 'medium', 2: 'high'},
    'Job_Satisfaction': {0: 'not satisfied', 1: 'satisfied', 2: 'very satisfied'},
    'Other_Job_Opportunities': {0: 'no', 1: 'yes'},
}

def map_value(feature_name, value):
    """Map encoded values to readable names"""
    if feature_name in FEATURE_MAPPINGS:
        mapping = FEATURE_MAPPINGS[feature_name]
        if value in mapping:
            return mapping[value]
        if isinstance(value, float):
            if int(value) in mapping:
                return mapping[int(value)]
    return str(value)


feature_desc = ['the gender of the employee', 
                'the age of the employee',
                'the highest obtained academic degree of the employee',
                'the total years of experience of the employee overall all their jobs',
                'the years of experience the employee has in this job',
                'the sector in which the employee is employed',
                'the monthly salary of the employee in SAR',
                'whether the emloyee has medical ensurance in this job',
                'whether the employee received an annual bonus for their performance in this job',
                'whether the employee had overtime in this job', 
                'whether the employee received compenstation for their possible overtime in this job',
                'whether the employee is satisfied with their income relative to their effort in this job',
                'whether the employee felt they got the promotion they deserved in this job',
                'the amount of training programs the employee attended in the last 3 years of this job',
                'whether the employee benefited from the training programs provided by their current employer',
                'how often the employee travels for business purposes in this job',
                'the level of support the employee receives from their organization in this job',
                'whether the employee feels moral appreciation and recognition of their effort by their superiors in this job',
                'the emotional commitment and psychological relationship of the employee with their current organization', 
                'how easy it was to get involved in the job (e.g. decision making, opinions) of the employee in this job', 
                'the distance to the employee his/her workplace in this job',
                'how easy it is to balance work life and personal life at work for the employee in this job',
                'whether the employee felt physically stressed due to physically demanding tasks in this job',
                'whether the employee felt a state of psychological exhaustion and mental or emotional fatigue as a result of their constant exposure to job stress in this job',
                'whether the employee feels a sense of job security and stability in this job',
                'whether the employee did have any health issues that forced them to leave this job',
                'the satisfaction of the employee with their work environment in this job',
                'the satisfaction of the employee with this job in general',
                'whether the employee received other job opportunities while working in this job'
]

# Build feature dataframe with feature names and descriptions only (like credit dataset)
feature_data = []
for i, col in enumerate(x_train.columns):
    row = {
        "feature_name": col,
        "feature_desc": feature_desc[i]
    }
    feature_data.append(row)

feature_desc_df = pd.DataFrame(feature_data)
     

dataset_description="data about the Saudi private sector to better understand the causes and consequences of employee satisfaction and turnover. The initial step to collect this data was administering an online survey to 1,200 employees of various Saudi Arabian companies. The dataset's question axes were described using a total of 29 qualities which you are given"
target_description="The target variable is whether the employee will leave their job (1) or not (0)."
task_description=" The ML model aims to predict whether an employee will leave the job or not."

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [10]:
dataset_info

{'dataset_description': "data about the Saudi private sector to better understand the causes and consequences of employee satisfaction and turnover. The initial step to collect this data was administering an online survey to 1,200 employees of various Saudi Arabian companies. The dataset's question axes were described using a total of 29 qualities which you are given",
 'target_description': 'The target variable is whether the employee will leave their job (1) or not (0).',
 'task_description': ' The ML model aims to predict whether an employee will leave the job or not.',
 'feature_description':                                  feature_name  \
 0                                      Gender   
 1                                         Age   
 2                             Academic_degree   
 3                            Years_Experience   
 4           Years_experience_lastorganization   
 5                                      Sector   
 6                               MonthlySalary 